# SmartTraffic — RR-TFormer
## Reliability-Regularized Transformer with Privileged Reliability Supervision

This is a **fixed-protocol redesign** that does not depend on CRR-TFormer.

### Principle
At inference time:
\[
X \rightarrow \text{Transformer} \rightarrow \hat Y
\]
Only camera-derived traffic history \(X\) enters the forecasting model. Visual reliability descriptors are **not** used as inference inputs.

During training, synthetic corrupted histories are generated only from the train/validation split. Reliability is used as **privileged supervision** for auxiliary denoising and observation-quality objectives.

### Training tasks
1. Clean forecasting  
2. Corruption-consistent forecasting with the same clean future target  
3. Denoising reconstruction of the clean historical traffic sequence  
4. Auxiliary reliability prediction from latent traffic representations  
5. Clean/corrupted forecast consistency

### Reliability target
For each historical step:
\[
r_t^*=\exp\left(-2\frac{\|X^{corrupt}_{t,veh}-X^{clean}_{t,veh}\|_1}{\|X^{clean}_{t,veh}\|_1+1}\right)
\]
The reliability target is used **only during training**.

### No expensive repetition
This notebook does **not** download AI City, run YOLO, run ByteTrack, regenerate clean video features, or regenerate controlled degradation video.

### Pre-registered decision rule
**Strong success**: clean mean MAE < 11.0902 and average paired relative degradation < 8.542%.  
**Publishable RR success**: clean mean MAE <= 11.20, average relative degradation <= 10%, 120-s MAE <= 11.4349, and no individual degradation scenario exceeds 40% relative degradation.  
Otherwise stop architecture optimization.

In [ ]:
# 0. COLAB SETUP
%pip -q install pyarrow openpyxl statsmodels scipy scikit-learn tqdm tabulate
from google.colab import drive
drive.mount('/content/drive')
import os, math, json, time, random, zipfile, warnings, re
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')
print('Drive mounted. No dataset download or video reprocessing will be performed.')

In [ ]:
# 1. CONFIG + PATHS
CFG={
 'SEED':42,'PROJECT_NAME':'SmartTraffic_VRA_TFormer_IJIES',
 'BIN_SEC':5,'LOOKBACK_STEPS':60,'FORECAST_BLOCK_STEPS':6,'N_HORIZONS':4,
 'TRAIN_FRAC':0.60,'VAL_FRAC':0.20,
 'D_MODEL':64,'N_HEAD':4,'N_LAYERS':2,'DROPOUT':0.10,
 'EPOCHS':80,'PATIENCE':12,'BATCH_SIZE':64,
 'LR_BACKBONE':2e-4,'LR_AUX':8e-4,'WEIGHT_DECAY':1e-4,
 'AUG_COPIES_TRAIN':3,'AUG_COPIES_VAL':1,
 'W_FORECAST_CLEAN':1.0,'W_FORECAST_CORRUPT':1.0,'W_RECON':0.30,
 'W_REL':0.15,'W_CONSISTENCY':0.05,'W_CLEAN_AUX':0.05,
 'ROBUST_LOOKBACK_STEPS':24,'FIG_DPI':400}
SEED=CFG['SEED']; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:',DEVICE)
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
ROOT=Path('/content/drive/MyDrive')/CFG['PROJECT_NAME']
D={'root':ROOT,'cache':ROOT/'cache','clean':ROOT/'cache'/'aicity_clean','degraded':ROOT/'cache'/'aicity_degraded','models':ROOT/'models','rr':ROOT/'rr'}
D['rr'].mkdir(parents=True,exist_ok=True)
if not ROOT.exists(): raise FileNotFoundError(f'Project folder not found: {ROOT}')

In [ ]:
# 2. LOAD EXISTING CACHES
def load_cache(aggregate_path,parts_root):
 aggregate_path=Path(aggregate_path)
 if aggregate_path.exists():
  print('[cache]',aggregate_path.name); return pd.read_parquet(aggregate_path)
 parts=sorted(Path(parts_root).rglob('*.parquet'))
 if not parts: raise RuntimeError(f'No parquet cache found under {parts_root}')
 df=pd.concat([pd.read_parquet(p) for p in parts],ignore_index=True); df.to_parquet(aggregate_path,index=False); return df
clean_df=load_cache(D['cache']/'aicity_clean_all.parquet',D['clean'])
degraded_df=load_cache(D['cache']/'aicity_degraded_all.parquet',D['degraded'])
clean_df=clean_df.sort_values(['sequence','bin_idx']).reset_index(drop=True)
degraded_df=degraded_df.sort_values(['condition','severity','sequence','bin_idx']).reset_index(drop=True)
TRAFFIC_COLS=['car','motorcycle','bus','truck','total']
REL_COLS=['mean_conf','track_continuity','blur_log','illumination_quality','occlusion_quality','frame_availability']
for c in TRAFFIC_COLS+REL_COLS:
 if c not in clean_df.columns: raise RuntimeError(f'Missing clean-cache column: {c}')
print('Clean sequences:',clean_df['sequence'].nunique())
print('Degraded sequences:',degraded_df['sequence'].nunique())
print('Degradation scenarios:',degraded_df[['condition','severity']].drop_duplicates().shape[0])

In [ ]:
# 3. EXACT SPLIT + TRAIN-ONLY SCALERS
from sklearn.preprocessing import StandardScaler
for c in REL_COLS: clean_df[c]=clean_df.groupby('sequence')[c].transform(lambda s:s.interpolate(limit_direction='both'))
bounds={}
for seq,g in clean_df.groupby('sequence'):
 n=len(g); tr=int(n*CFG['TRAIN_FRAC']); va=int(n*(CFG['TRAIN_FRAC']+CFG['VAL_FRAC'])); bounds[seq]=(tr,va,n)
train_row_mask=np.zeros(len(clean_df),dtype=bool)
for seq,idx in clean_df.groupby('sequence').groups.items():
 ids=list(idx); train_row_mask[ids[:bounds[seq][0]]]=True
rel_median=clean_df.loc[train_row_mask,REL_COLS].median()
for c in REL_COLS: clean_df[c]=clean_df[c].fillna(rel_median[c])
x_scaler=StandardScaler().fit(clean_df.loc[train_row_mask,TRAFFIC_COLS])
q_scaler=StandardScaler().fit(clean_df.loc[train_row_mask,REL_COLS])
REL_LOW=clean_df.loc[train_row_mask,REL_COLS].quantile(0.05).to_dict()
L=CFG['LOOKBACK_STEPS']; B=CFG['FORECAST_BLOCK_STEPS']; H=CFG['N_HORIZONS']; FUTURE=B*H
samples=[]
for seq,g in clean_df.groupby('sequence',sort=False):
 g=g.reset_index(drop=True); tr,va,n=bounds[seq]
 X_raw=g[TRAFFIC_COLS].values.astype(np.float32); Q_raw=g[REL_COLS].values.astype(np.float32)
 X=x_scaler.transform(g[TRAFFIC_COLS]).astype(np.float32); Q=q_scaler.transform(g[REL_COLS]).astype(np.float32)
 y_raw_series=g['total'].values.astype(float)
 for t in range(L,n-FUTURE+1):
  y_raw=np.array([y_raw_series[t+h*B:t+(h+1)*B].sum() for h in range(H)],dtype=np.float32)
  target_end=t+FUTURE
  if target_end<=tr: split='train'
  elif t>=tr and target_end<=va: split='val'
  elif t>=va: split='test'
  else: continue
  samples.append({'sequence':seq,'split':split,'t':int(t),'x':X[t-L:t],'q':Q[t-L:t],'x_raw':X_raw[t-L:t],'q_raw':Q_raw[t-L:t],'y_raw':y_raw})
train_targets=np.vstack([r['y_raw'] for r in samples if r['split']=='train'])
y_scaler=StandardScaler().fit(train_targets.reshape(-1,1))
for r in samples: r['y']=y_scaler.transform(r['y_raw'].reshape(-1,1)).reshape(-1).astype(np.float32)
split_summary=pd.DataFrame({'split':['train','val','test'],'windows':[sum(r['split']==s for r in samples) for s in ['train','val','test']]})
display(split_summary)
expected={'train':798,'val':294,'test':294}; observed=dict(zip(split_summary['split'],split_summary['windows']))
if observed!=expected: raise RuntimeError(f'Split mismatch. Expected {expected}, got {observed}.')
print('✅ Exact split match.')

In [ ]:
# 4. BASELINE MODELS + CHECKPOINTS
NT=len(TRAFFIC_COLS); NR=len(REL_COLS); NO=CFG['N_HORIZONS']; DM=CFG['D_MODEL']
class PositionalEncoding(nn.Module):
 def __init__(self,d_model,max_len=2048):
  super().__init__(); pe=torch.zeros(max_len,d_model); pos=torch.arange(max_len).unsqueeze(1).float(); div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model)); pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div); self.register_buffer('pe',pe.unsqueeze(0),persistent=False)
 def forward(self,x): return x+self.pe[:,:x.size(1)]
class TransformerModel(nn.Module):
 def __init__(self,input_dim=NT):
  super().__init__(); self.p=nn.Linear(input_dim,DM); self.pos=PositionalEncoding(DM)
  layer=nn.TransformerEncoderLayer(DM,CFG['N_HEAD'],DM*4,CFG['DROPOUT'],batch_first=True,norm_first=True); self.e=nn.TransformerEncoder(layer,CFG['N_LAYERS']); self.h=nn.Sequential(nn.LayerNorm(DM),nn.Linear(DM,NO))
 def encode(self,x): return self.e(self.pos(self.p(x)))
 def forward(self,x,q=None): z=self.encode(x); return self.h(z.mean(dim=1))
class ReliabilityConcatModel(nn.Module):
 def __init__(self): super().__init__(); self.m=TransformerModel(NT+NR)
 def forward(self,x,q=None): return self.m(torch.cat([x,q],dim=-1))
class ReliabilityAttentionLayer(nn.Module):
 def __init__(self):
  super().__init__(); self.nh=CFG['N_HEAD']; self.att=nn.MultiheadAttention(DM,self.nh,dropout=CFG['DROPOUT'],batch_first=True); self.n1=nn.LayerNorm(DM); self.n2=nn.LayerNorm(DM); self.ff=nn.Sequential(nn.Linear(DM,DM*4),nn.GELU(),nn.Dropout(CFG['DROPOUT']),nn.Linear(DM*4,DM))
 def forward(self,x,reliability=None,lam=None):
  mask=None
  if reliability is not None and lam is not None:
   b0,l0,_=reliability.shape; key_bias=lam*torch.log(reliability.squeeze(-1).clamp_min(1e-5)); mask=key_bias[:,None,:].expand(b0,l0,l0).repeat_interleave(self.nh,dim=0)
  h=self.n1(x); a,_=self.att(h,h,h,attn_mask=mask,need_weights=False); x=x+a; return x+self.ff(self.n2(x))
class OldVRATFormer(nn.Module):
 def __init__(self):
  super().__init__(); self.obs=nn.Sequential(nn.Linear(NT,DM),nn.GELU()); self.rel=nn.Sequential(nn.Linear(NR,DM//2),nn.GELU()); self.gate=nn.Sequential(nn.Linear(DM//2,1),nn.Sigmoid()); self.ctx=nn.GRU(NT,DM,batch_first=True); self.pos=PositionalEncoding(DM); self.layers=nn.ModuleList([ReliabilityAttentionLayer() for _ in range(CFG['N_LAYERS'])]); self.rawlam=nn.Parameter(torch.tensor(0.0)); self.head=nn.Sequential(nn.LayerNorm(DM),nn.Linear(DM,NO))
 def forward(self,x,q=None):
  obs=self.obs(x); ctx,_=self.ctx(x); r=self.gate(self.rel(q)); z=self.pos(r*obs+(1-r)*ctx); lam=F.softplus(self.rawlam)
  for layer in self.layers: z=layer(z,r,lam)
  return self.head(z.mean(dim=1))
def safe_model_name(name): return re.sub(r'[^A-Za-z0-9_-]+','_',name)
def ckpt(name): return D['models']/f'{safe_model_name(name)}.pt'
def load_state(path):
 try: return torch.load(path,map_location=DEVICE,weights_only=True)
 except TypeError: return torch.load(path,map_location=DEVICE)
baseline_transformer=TransformerModel().to(DEVICE); baseline_concat=ReliabilityConcatModel().to(DEVICE); old_vra=OldVRATFormer().to(DEVICE)
required={'Transformer':ckpt('Transformer'),'Transformer+ReliabilityConcat':ckpt('Transformer+ReliabilityConcat'),'VRA-TFormer':ckpt('VRA-TFormer')}
for name,path in required.items(): print(name,'✅' if path.exists() else '❌',path)
if not all(p.exists() for p in required.values()): raise RuntimeError('Required completed-run baseline checkpoint is missing.')
baseline_transformer.load_state_dict(load_state(required['Transformer'])); baseline_concat.load_state_dict(load_state(required['Transformer+ReliabilityConcat'])); old_vra.load_state_dict(load_state(required['VRA-TFormer']))
for model in [baseline_transformer,baseline_concat,old_vra]:
 model.eval();
 for p in model.parameters(): p.requires_grad=False

In [ ]:
# 5. RR-TFORMER — INFERENCE PATH = TRAFFIC ONLY
class RRTFormer(nn.Module):
 def __init__(self,transformer_state):
  super().__init__(); self.backbone=TransformerModel(); self.backbone.load_state_dict(transformer_state)
  self.reconstruction_head=nn.Sequential(nn.Linear(DM,DM),nn.GELU(),nn.Linear(DM,NT))
  self.reliability_head=nn.Sequential(nn.Linear(DM,32),nn.GELU(),nn.Linear(32,1),nn.Sigmoid())
 def forward(self,x,return_aux=False):
  z=self.backbone.encode(x); pred=self.backbone.h(z.mean(dim=1))
  if not return_aux: return pred
  return {'pred':pred,'latent':z,'reconstructed':self.reconstruction_head(z),'reliability':self.reliability_head(z)}
rr=RRTFormer(load_state(required['Transformer'])).to(DEVICE)
print('RR total parameters:',f"{sum(p.numel() for p in rr.parameters()):,}")
print('RR inference parameters:',f"{sum(p.numel() for p in rr.backbone.parameters()):,}")

In [ ]:
# 6. SYNTHETIC TRAIN/VAL CORRUPTION — RELIABILITY IS TARGET ONLY
def synth_corrupt_raw(x_clean_raw,q_clean_raw,rng):
 x=x_clean_raw.copy().astype(np.float32); q=q_clean_raw.copy().astype(np.float32); severity=float(rng.uniform(0.20,0.85)); condition=str(rng.choice(['blur','lowlight','occlusion','frame_drop']))
 def degrade_rel(col,strength):
  j=REL_COLS.index(col); low=float(REL_LOW[col]); q[:,j]=(1-strength)*q[:,j]+strength*low
 if condition=='blur':
  degrade_rel('blur_log',severity); degrade_rel('mean_conf',0.70*severity); degrade_rel('track_continuity',0.45*severity); x[:,:4]*=(1.0-0.35*severity)
 elif condition=='lowlight':
  degrade_rel('illumination_quality',severity); degrade_rel('mean_conf',0.75*severity); degrade_rel('track_continuity',0.35*severity); x[:,:4]*=(1.0-0.30*severity)
 elif condition=='occlusion':
  degrade_rel('occlusion_quality',severity); degrade_rel('track_continuity',0.65*severity); degrade_rel('mean_conf',0.35*severity); x[:,:4]*=(1.0-0.38*severity); mask=rng.random(len(x))<(0.08+0.18*severity); x[mask,:4]*=rng.uniform(0.20,max(0.25,1-severity))
 else:
  degrade_rel('frame_availability',severity); degrade_rel('track_continuity',0.60*severity); drop_p=min(0.50,0.55*severity); mask=rng.random(len(x))<drop_p; x[mask,:4]*=rng.uniform(0.0,max(0.10,1-severity)); x[~mask,:4]*=(1.0-0.10*severity)
 noise_sd=0.02+0.06*severity; x[:,:4]*=np.clip(rng.normal(1.0,noise_sd,size=x[:,:4].shape),0.0,None); x[:,:4]=np.clip(x[:,:4],0.0,None); x[:,4]=x[:,:4].sum(axis=1)
 numerator=np.abs(x_clean_raw[:,:4]-x[:,:4]).sum(axis=1); denominator=np.abs(x_clean_raw[:,:4]).sum(axis=1)+1.0; r_target=np.exp(-2.0*(numerator/denominator)).astype(np.float32); r_target=np.clip(r_target,0.0,1.0)[:,None]
 return x_scaler.transform(x).astype(np.float32),r_target,severity,condition
def build_rr_records(rows,split,copies):
 source=[r for r in rows if r['split']==split]; out=[]
 for i,r in enumerate(source):
  for k in range(copies):
   rng=np.random.default_rng(SEED+i*1009+k*7919+(0 if split=='train' else 900000)); xc,rt,sev,cond=synth_corrupt_raw(r['x_raw'],r['q_raw'],rng)
   out.append({'x_clean':r['x'].astype(np.float32),'x_corrupt':xc,'r_target':rt,'y':r['y'].astype(np.float32),'severity':np.float32(sev),'condition':cond})
 return out
train_records=build_rr_records(samples,'train',CFG['AUG_COPIES_TRAIN']); val_records=build_rr_records(samples,'val',CFG['AUG_COPIES_VAL'])
print('RR train pairs:',len(train_records),'RR val pairs:',len(val_records))
class RRTrainDS(Dataset):
 def __init__(self,records): self.records=records
 def __len__(self): return len(self.records)
 def __getitem__(self,i):
  r=self.records[i]; return torch.tensor(r['x_clean'],dtype=torch.float32),torch.tensor(r['x_corrupt'],dtype=torch.float32),torch.tensor(r['r_target'],dtype=torch.float32),torch.tensor(r['y'],dtype=torch.float32)
train_loader=DataLoader(RRTrainDS(train_records),batch_size=CFG['BATCH_SIZE'],shuffle=True,num_workers=0); val_loader=DataLoader(RRTrainDS(val_records),batch_size=CFG['BATCH_SIZE'],shuffle=False,num_workers=0)

In [ ]:
# 7. ROBUSTNESS-REGULARIZED ROBUSTNESS-REGULARIZED FINE-TUNING
optimizer=torch.optim.AdamW([{'params':rr.backbone.parameters(),'lr':CFG['LR_BACKBONE']},{'params':rr.reconstruction_head.parameters(),'lr':CFG['LR_AUX']},{'params':rr.reliability_head.parameters(),'lr':CFG['LR_AUX']}],weight_decay=CFG['WEIGHT_DECAY'])
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=0.5,patience=5)
def rr_loss(batch):
 x_clean,x_corrupt,r_target,y=batch; x_clean=x_clean.to(DEVICE); x_corrupt=x_corrupt.to(DEVICE); r_target=r_target.to(DEVICE); y=y.to(DEVICE)
 co=rr(x_clean,return_aux=True); xo=rr(x_corrupt,return_aux=True)
 l_fc=F.mse_loss(co['pred'],y); l_fcor=F.mse_loss(xo['pred'],y); l_recon=F.smooth_l1_loss(xo['reconstructed'],x_clean); l_rel=F.smooth_l1_loss(xo['reliability'],r_target)
 l_clean_aux=F.smooth_l1_loss(co['reconstructed'],x_clean)+F.smooth_l1_loss(co['reliability'],torch.ones_like(co['reliability'])); l_cons=F.smooth_l1_loss(xo['pred'],co['pred'].detach())
 total=CFG['W_FORECAST_CLEAN']*l_fc+CFG['W_FORECAST_CORRUPT']*l_fcor+CFG['W_RECON']*l_recon+CFG['W_REL']*l_rel+CFG['W_CONSISTENCY']*l_cons+CFG['W_CLEAN_AUX']*l_clean_aux
 return total,{'loss':float(total.detach().cpu()),'forecast_clean':float(l_fc.detach().cpu()),'forecast_corrupt':float(l_fcor.detach().cpu()),'recon':float(l_recon.detach().cpu()),'reliability':float(l_rel.detach().cpu()),'consistency':float(l_cons.detach().cpu()),'r_clean':float(co['reliability'].mean().detach().cpu()),'r_corrupt':float(xo['reliability'].mean().detach().cpu())}
def run_epoch(loader,training):
 rr.train(training); rows=[]; context=torch.enable_grad() if training else torch.no_grad()
 with context:
  for batch in loader:
   if training: optimizer.zero_grad(set_to_none=True)
   loss,m=rr_loss(batch)
   if training: loss.backward(); torch.nn.utils.clip_grad_norm_(rr.parameters(),1.0); optimizer.step()
   rows.append(m)
 return pd.DataFrame(rows).mean().to_dict()
best_val=float('inf'); wait=0; history=[]; rr_ckpt=D['rr']/'RR-TFormer_FINAL.pt'
for epoch in range(1,CFG['EPOCHS']+1):
 tr=run_epoch(train_loader,True); va=run_epoch(val_loader,False); scheduler.step(va['loss']); history.append({'epoch':epoch,**{f'train_{k}':v for k,v in tr.items()},**{f'val_{k}':v for k,v in va.items()},'lr_backbone':optimizer.param_groups[0]['lr'],'lr_aux':optimizer.param_groups[1]['lr']})
 if epoch==1 or epoch%5==0: print(f"epoch={epoch:03d} train={tr['loss']:.4f} val={va['loss']:.4f} fc={va['forecast_clean']:.4f} fcor={va['forecast_corrupt']:.4f} r_clean={va['r_clean']:.3f} r_corr={va['r_corrupt']:.3f}")
 if va['loss']<best_val-1e-6: best_val=va['loss']; wait=0; torch.save({'model':rr.state_dict(),'config':CFG},rr_ckpt)
 else:
  wait+=1
  if wait>=CFG['PATIENCE']: print('Early stopping.'); break
state=torch.load(rr_ckpt,map_location=DEVICE); rr.load_state_dict(state['model']); history_df=pd.DataFrame(history); history_df.to_csv(D['rr']/'RR_training_history.csv',index=False); print('✅ RR-TFormer fine-tuning complete.')

In [ ]:
# 8. CLEAN TEST EVALUATION
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
def inverse_target(a):
 a=np.asarray(a); shape=a.shape; return y_scaler.inverse_transform(a.reshape(-1,1)).reshape(shape)
class CleanDS(Dataset):
 def __init__(self,rows,split): self.rows=[r for r in rows if r['split']==split]
 def __len__(self): return len(self.rows)
 def __getitem__(self,i):
  r=self.rows[i]; return torch.tensor(r['x'],dtype=torch.float32),torch.tensor(r['q'],dtype=torch.float32),torch.tensor(r['y'],dtype=torch.float32),r['sequence'],int(r['t'])
test_loader=DataLoader(CleanDS(samples,'test'),batch_size=128,shuffle=False)
@torch.no_grad()
def predict(model,loader,model_type='plain'):
 model.eval(); ys=[]; ps=[]; aux_rel=[]
 for x,q,y,_,_ in loader:
  x=x.to(DEVICE); q=q.to(DEVICE)
  if model_type=='rr': out=model(x,return_aux=True); p=out['pred']; aux_rel.append(out['reliability'].cpu().numpy())
  elif model_type in ('vra','concat'): p=model(x,q)
  else: p=model(x)
  ys.append(y.numpy()); ps.append(p.cpu().numpy())
 yt=inverse_target(np.concatenate(ys)); yp=inverse_target(np.concatenate(ps)); rel=np.concatenate(aux_rel) if aux_rel else None; return yt,yp,rel
def metric_rows(y,p,name):
 rows=[]
 for j in range(NO):
  yt=y[:,j]; yp=p[:,j]; rows.append({'Model':name,'Horizon':f'{30*(j+1)}s','MAE':mean_absolute_error(yt,yp),'RMSE':mean_squared_error(yt,yp)**0.5,'WAPE_%':np.abs(yt-yp).sum()/max(np.abs(yt).sum(),1e-9)*100,'R2':r2_score(yt,yp)})
 return pd.DataFrame(rows)
clean_store={}; clean_tabs=[]
for name,model,typ in [('Transformer',baseline_transformer,'plain'),('Transformer+ReliabilityConcat',baseline_concat,'concat'),('VRA-TFormer',old_vra,'vra'),('RR-TFormer',rr,'rr')]:
 yt,yp,rel=predict(model,test_loader,typ); clean_store[name]=(yt,yp,rel); clean_tabs.append(metric_rows(yt,yp,name))
clean_results=pd.concat(clean_tabs,ignore_index=True); clean_mean=clean_results.groupby('Model',as_index=False).agg(MAE=('MAE','mean'),RMSE=('RMSE','mean'),WAPE_pct=('WAPE_%','mean'),R2=('R2','mean')).sort_values('MAE'); display(clean_results); display(clean_mean)

In [ ]:
# 9. CLEAN WILCOXON-HOLM
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
yt_rr,yp_rr,_=clean_store['RR-TFormer']; stat_rows=[]
for comparator in ['Transformer','Transformer+ReliabilityConcat','VRA-TFormer']:
 yt_b,yp_b,_=clean_store[comparator]; temp=[]; pvals=[]
 for j in range(NO):
  e_rr=np.abs(yt_rr[:,j]-yp_rr[:,j]); e_b=np.abs(yt_b[:,j]-yp_b[:,j])
  try: stat,p=wilcoxon(e_rr,e_b,alternative='less')
  except Exception: stat,p=np.nan,1.0
  temp.append({'Comparator':comparator,'Horizon':f'{30*(j+1)}s','Comparator_MAE':e_b.mean(),'RR_MAE':e_rr.mean(),'RR_Improvement_%':(e_b.mean()-e_rr.mean())/max(e_b.mean(),1e-9)*100,'RR_win_rate_%':np.mean(e_rr<e_b)*100,'Wilcoxon_stat':stat,'p_raw':p}); pvals.append(p)
 reject,padj,_,_=multipletests(pvals,alpha=0.05,method='holm')
 for row,pa,rj in zip(temp,padj,reject): row['p_holm']=pa; row['significant_0.05']=rj; stat_rows.append(row)
clean_stats=pd.DataFrame(stat_rows); display(clean_stats)

In [ ]:
# 10. REAL DEGRADED-CACHE ROBUSTNESS — RR INFERENCE REMAINS TRAFFIC-ONLY
ROBUST_L=CFG['ROBUST_LOOKBACK_STEPS']; degraded_sequences=sorted(degraded_df['sequence'].unique()); seq_common_length={}
for seq in degraded_sequences:
 clean_bins=set(clean_df.loc[clean_df['sequence']==seq,'bin_idx'].astype(int)); lengths=[]
 for (_, _),g in degraded_df[degraded_df['sequence']==seq].groupby(['condition','severity']): lengths.append(len(clean_bins & set(g['bin_idx'].astype(int))))
 if lengths: seq_common_length[seq]=min(lengths)
robust_sequences=[s for s,n in seq_common_length.items() if n>=ROBUST_L+FUTURE+1]; print('Robustness sequences:',robust_sequences)
if not robust_sequences: raise RuntimeError('No existing degraded sequence supports the fixed protocol.')
def prep_frame(df0):
 g=df0.sort_values('bin_idx').drop_duplicates('bin_idx').copy()
 for c in REL_COLS: g[c]=g[c].interpolate(limit_direction='both').fillna(rel_median[c])
 return g
def build_pair(scenario_df,seq):
 deg=prep_frame(scenario_df[scenario_df['sequence']==seq]); ref=prep_frame(clean_df[clean_df['sequence']==seq]); common=sorted(set(deg['bin_idx'].astype(int)) & set(ref['bin_idx'].astype(int)))
 if not common: return [],[]
 deg=deg[deg['bin_idx'].isin(common)].sort_values('bin_idx').reset_index(drop=True); ref=ref[ref['bin_idx'].isin(common)].sort_values('bin_idx').reset_index(drop=True); n=min(len(deg),len(ref)); deg=deg.iloc[:n].copy(); ref=ref.iloc[:n].copy(); xd=x_scaler.transform(deg[TRAFFIC_COLS]).astype(np.float32); qd=q_scaler.transform(deg[REL_COLS]).astype(np.float32); xc=x_scaler.transform(ref[TRAFFIC_COLS]).astype(np.float32); qc=q_scaler.transform(ref[REL_COLS]).astype(np.float32); clean_total=ref['total'].values.astype(float); dr=[]; cr=[]
 for t in range(ROBUST_L,n-FUTURE+1):
  y_raw=np.array([clean_total[t+h*B:t+(h+1)*B].sum() for h in range(H)],dtype=np.float32); y=y_scaler.transform(y_raw.reshape(-1,1)).reshape(-1).astype(np.float32); base={'y':y,'y_raw':y_raw}; dr.append({**base,'x':xd[t-ROBUST_L:t],'q':qd[t-ROBUST_L:t]}); cr.append({**base,'x':xc[t-ROBUST_L:t],'q':qc[t-ROBUST_L:t]})
 return dr,cr
class PairDS(Dataset):
 def __init__(self,rows): self.rows=rows
 def __len__(self): return len(self.rows)
 def __getitem__(self,i):
  r=self.rows[i]; return torch.tensor(r['x'],dtype=torch.float32),torch.tensor(r['q'],dtype=torch.float32),torch.tensor(r['y'],dtype=torch.float32),'seq',i
def pair_loader(rows): return DataLoader(PairDS(rows),batch_size=128,shuffle=False)
rob_models={'Transformer':(baseline_transformer,'plain'),'Transformer+ReliabilityConcat':(baseline_concat,'concat'),'VRA-TFormer':(old_vra,'vra'),'RR-TFormer':(rr,'rr')}; robust_rows=[]; robust_store={}
for (condition,severity),scenario_df in degraded_df.groupby(['condition','severity']):
 dr_all=[]; cr_all=[]
 for seq in robust_sequences:
  dr,cr=build_pair(scenario_df,seq); dr_all.extend(dr); cr_all.extend(cr)
 if not dr_all: continue
 dl=pair_loader(dr_all); cl=pair_loader(cr_all); print(condition,severity,'windows',len(dr_all))
 for name,(model,typ) in rob_models.items():
  yc,pc,rc=predict(model,cl,typ); yd,pd0,rd=predict(model,dl,typ)
  if not np.allclose(yc,yd): raise RuntimeError('Paired target mismatch.')
  clean_mae=np.abs(yc-pc).mean(); deg_mae=np.abs(yd-pd0).mean(); relative_deg=(deg_mae-clean_mae)/max(clean_mae,1e-9)*100
  robust_rows.append({'Model':name,'Condition':condition,'Severity':int(severity),'Clean_MAE':clean_mae,'Degraded_MAE':deg_mae,'Relative_Degradation_%':relative_deg,'N_windows':len(dr_all),'N_sequences':len(robust_sequences),'Aux_Reliability_Clean':float(rc.mean()) if rc is not None else np.nan,'Aux_Reliability_Degraded':float(rd.mean()) if rd is not None else np.nan})
  robust_store[(condition,int(severity),name)]={'y':yd,'pred':pd0,'window_error':np.abs(yd-pd0).mean(axis=1),'aux_reliability':rd}
robust_summary=pd.DataFrame(robust_rows); avg_robust=robust_summary.groupby('Model',as_index=False).agg(Clean_MAE=('Clean_MAE','mean'),Degraded_MAE=('Degraded_MAE','mean'),Relative_Degradation_pct=('Relative_Degradation_%','mean')).sort_values('Relative_Degradation_pct'); display(robust_summary); display(avg_robust)

In [ ]:
# 11. ROBUSTNESS STATISTICS
robust_stat_rows=[]; scenarios=robust_summary[['Condition','Severity']].drop_duplicates().sort_values(['Condition','Severity'])
for condition,severity in scenarios.itertuples(index=False,name=None):
 rr_err=robust_store[(condition,int(severity),'RR-TFormer')]['window_error']
 for comparator in ['Transformer','Transformer+ReliabilityConcat','VRA-TFormer']:
  base_err=robust_store[(condition,int(severity),comparator)]['window_error']; n=min(len(rr_err),len(base_err)); re0=rr_err[:n]; be0=base_err[:n]
  try: stat,p=wilcoxon(re0,be0,alternative='less')
  except Exception: stat,p=np.nan,1.0
  robust_stat_rows.append({'Condition':condition,'Severity':int(severity),'Comparator':comparator,'N_windows':n,'Comparator_MAE':be0.mean(),'RR_MAE':re0.mean(),'RR_Improvement_%':(be0.mean()-re0.mean())/max(be0.mean(),1e-9)*100,'RR_win_rate_%':np.mean(re0<be0)*100,'Wilcoxon_stat':stat,'p_raw':p})
robust_stats=pd.DataFrame(robust_stat_rows); pieces=[]
for comparator,g in robust_stats.groupby('Comparator'):
 g=g.copy(); reject,padj,_,_=multipletests(g['p_raw'].values,alpha=0.05,method='holm'); g['p_holm']=padj; g['significant_0.05']=reject; pieces.append(g)
robust_stats=pd.concat(pieces,ignore_index=True); display(robust_stats)

In [ ]:
# 12. PRE-REGISTERED DECISION
clean_lookup=clean_mean.set_index('Model'); rob_lookup=avg_robust.set_index('Model'); rr_clean=float(clean_lookup.loc['RR-TFormer','MAE']); rr_rd=float(rob_lookup.loc['RR-TFormer','Relative_Degradation_pct']); h120=clean_results[clean_results['Horizon']=='120s'].set_index('Model')['MAE']; rr_120=float(h120['RR-TFormer']); rr_scenarios=robust_summary[robust_summary['Model']=='RR-TFormer']; max_scenario_rd=float(rr_scenarios['Relative_Degradation_%'].max())
BEST_CLEAN_REFERENCE=11.0902; BEST_RD_REFERENCE=8.5420; VANILLA_120_REFERENCE=11.4349
strict_clean=rr_clean<BEST_CLEAN_REFERENCE; strict_robust=rr_rd<BEST_RD_REFERENCE; publishable_clean=rr_clean<=11.20; publishable_robust=rr_rd<=10.0; h120_ok=rr_120<=VANILLA_120_REFERENCE; scenario_guardrail=max_scenario_rd<=40.0
decision=pd.DataFrame([['Strict clean best',strict_clean,rr_clean,BEST_CLEAN_REFERENCE],['Strict robustness best',strict_robust,rr_rd,BEST_RD_REFERENCE],['Clean MAE <=11.20',publishable_clean,rr_clean,11.20],['Average robustness degradation <=10%',publishable_robust,rr_rd,10.0],['120-s MAE <= vanilla reference',h120_ok,rr_120,VANILLA_120_REFERENCE],['Worst scenario degradation <=40%',scenario_guardrail,max_scenario_rd,40.0]],columns=['Criterion','Pass','RR_value','Threshold_reference']); display(decision)
if strict_clean and strict_robust: FINAL_DECISION='STRONG SUCCESS: RR-TFormer achieves the strongest result on both clean mean MAE and average paired robustness.'
elif publishable_clean and publishable_robust and h120_ok and scenario_guardrail: FINAL_DECISION='PUBLISHABLE RR SUCCESS: traffic-only inference retains clean accuracy while robustness is near or better than the vanilla Transformer without direct reliability-feature exposure at inference.'
else: FINAL_DECISION='STOP ARCHITECTURE OPTIMIZATION: RR-TFormer did not satisfy the pre-registered success criteria. Retain the existing accuracy-robustness manuscript.'
print('\n============================================'); print(FINAL_DECISION); print('============================================')

In [ ]:
# 13. SAVE RR PAPER ARTIFACTS
OUTDIR=D['rr']; clean_results.to_csv(OUTDIR/'RR_clean_results.csv',index=False); clean_mean.to_csv(OUTDIR/'RR_clean_mean.csv',index=False); clean_stats.to_csv(OUTDIR/'RR_clean_wilcoxon.csv',index=False); robust_summary.to_csv(OUTDIR/'RR_robustness_summary.csv',index=False); avg_robust.to_csv(OUTDIR/'RR_robustness_average.csv',index=False); robust_stats.to_csv(OUTDIR/'RR_robustness_wilcoxon.csv',index=False); decision.to_csv(OUTDIR/'RR_decision.csv',index=False)
xlsx_out=OUTDIR/'SmartTraffic_RR_TFormer_Results.xlsx'
with pd.ExcelWriter(xlsx_out,engine='openpyxl') as writer:
 split_summary.to_excel(writer,sheet_name='Split',index=False); clean_results.to_excel(writer,sheet_name='Clean_Horizon',index=False); clean_mean.to_excel(writer,sheet_name='Clean_Mean',index=False); clean_stats.to_excel(writer,sheet_name='Clean_Wilcoxon',index=False); robust_summary.to_excel(writer,sheet_name='Robustness',index=False); avg_robust.to_excel(writer,sheet_name='Robust_Average',index=False); robust_stats.to_excel(writer,sheet_name='Robust_Wilcoxon',index=False); decision.to_excel(writer,sheet_name='Decision',index=False)
fig,ax=plt.subplots(figsize=(7.3,4.0))
for name in ['Transformer','Transformer+ReliabilityConcat','VRA-TFormer','RR-TFormer']:
 z=clean_results[clean_results['Model']==name]; ax.plot(z['Horizon'],z['MAE'],marker='o',label=name)
ax.set_xlabel('Forecast horizon'); ax.set_ylabel('MAE (vehicles / 30-s block)'); ax.set_title('RR-TFormer clean multi-horizon comparison'); ax.legend(frameon=False); fig.tight_layout(); fig.savefig(OUTDIR/'RR_clean_MAE.png',dpi=CFG['FIG_DPI'],bbox_inches='tight'); fig.savefig(OUTDIR/'RR_clean_MAE.pdf',bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(7.0,3.8)); z=avg_robust.sort_values('Relative_Degradation_pct'); ax.bar(z['Model'],z['Relative_Degradation_pct']); ax.set_ylabel('Average relative MAE degradation (%)'); ax.set_title('RR-TFormer controlled robustness'); ax.tick_params(axis='x',rotation=20); fig.tight_layout(); fig.savefig(OUTDIR/'RR_average_robustness.png',dpi=CFG['FIG_DPI'],bbox_inches='tight'); fig.savefig(OUTDIR/'RR_average_robustness.pdf',bbox_inches='tight'); plt.show()
rr_rob=robust_summary[robust_summary['Model']=='RR-TFormer'].copy(); rr_rob['Scenario']=rr_rob['Condition'].astype(str)+'-S'+rr_rob['Severity'].astype(str); fig,ax=plt.subplots(figsize=(8.0,3.8)); x=np.arange(len(rr_rob)); w=0.35; ax.bar(x-w/2,rr_rob['Aux_Reliability_Clean'],width=w,label='Matched clean'); ax.bar(x+w/2,rr_rob['Aux_Reliability_Degraded'],width=w,label='Degraded'); ax.set_xticks(x); ax.set_xticklabels(rr_rob['Scenario'],rotation=25); ax.set_ylim(0,1); ax.set_ylabel('Auxiliary predicted reliability'); ax.set_title('RR auxiliary reliability response on real degradation'); ax.legend(frameon=False); fig.tight_layout(); fig.savefig(OUTDIR/'RR_aux_reliability_real_degradation.png',dpi=CFG['FIG_DPI'],bbox_inches='tight'); fig.savefig(OUTDIR/'RR_aux_reliability_real_degradation.pdf',bbox_inches='tight'); plt.show()
digest=['# SmartTraffic RR-TFormer — Robustness-Regularized Digest','','## Architecture','- Inference input: traffic history only.','- Visual reliability is privileged training supervision only.','- Auxiliary denoising reconstruction head.','- Auxiliary reliability prediction head.','- Auxiliary heads are not needed for forecasting inference.','- No video or detector reprocessing.','','## Clean mean results',clean_mean.to_markdown(index=False),'','## Clean horizon results',clean_results.to_markdown(index=False),'','## Average paired robustness',avg_robust.to_markdown(index=False),'','## RR robustness details',rr_rob.to_markdown(index=False),'','## Decision criteria',decision.to_markdown(index=False),'','## Final decision',FINAL_DECISION]
digest_path=OUTDIR/'SmartTraffic_RR_TFormer_Digest.md'; digest_path.write_text('\n'.join(digest),encoding='utf-8'); zip_out=OUTDIR/'SmartTraffic_RR_TFormer_Artifacts.zip'
with zipfile.ZipFile(zip_out,'w',zipfile.ZIP_DEFLATED) as zf:
 for p in OUTDIR.iterdir():
  if p.is_file() and p.name!=zip_out.name: zf.write(p,arcname=p.name)
print('\n============================================'); print('RR-TFORMER EXPERIMENT COMPLETE'); print('============================================'); print('Excel :',xlsx_out); print('Digest:',digest_path); print('ZIP   :',zip_out); print('\nUPLOAD THESE THREE FILES BACK TO CHAT.')